<div align="center">

# 🚀 LaunchMesh · Image to 3D
### Powered by TripoSG · Free · No GPU required on your machine

**How to use:**
1. Click **Runtime → Run all** (or press `Ctrl+F9`)
2. Wait for the app to launch (~3–5 minutes first time)
3. Use the link that appears to open the LaunchMesh UI
4. Upload your image and generate!

---
</div>

In [ ]:
#@title Step 1 - Install dependencies { display-mode: "form" }
print("Installing dependencies... (3-5 min first time)")
import subprocess

def run(cmd, label=""):
    if label: print("  -> " + label)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print("  [!] " + r.stderr[-300:])
    return r

# Clone TripoSG
run("git clone --depth 1 https://github.com/VAST-AI-Research/TripoSG.git /content/TripoSG 2>/dev/null || true", "Cloning TripoSG")

# PyTorch first - diso needs it at build time
run("pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118", "Installing PyTorch")

# Pin huggingface_hub to version that still has cached_download
run("pip install -q huggingface_hub==0.21.4", "Installing huggingface_hub")

# Pin diffusers to compatible version
run("pip install -q diffusers==0.25.1 transformers==4.38.2 accelerate==0.27.2", "Installing diffusers stack")

# diso needs --no-build-isolation to find torch
run("pip install -q diso --no-build-isolation", "Installing diso")

# Other deps
run("pip install -q flask flask-cors einops trimesh omegaconf scikit-image", "Installing utils")
run("pip install -q pyngrok", "Installing pyngrok")

# TripoSG requirements
run("pip install -q -r /content/TripoSG/requirements.txt 2>/dev/null || true", "Installing TripoSG requirements")

print("")
print("[OK] All dependencies installed!")


In [ ]:
#@title  Step 2 — Download TripoSG model weights { display-mode: "form" }
from huggingface_hub import snapshot_download
import os

MODEL_DIR = '/content/checkpoints/TripoSG'
RMBG_DIR  = '/content/checkpoints/RMBG-1.4'

if not os.path.exists(MODEL_DIR):
    print('Downloading TripoSG weights (~2 GB)...')
    snapshot_download('VAST-AI/TripoSG', local_dir=MODEL_DIR)
    print('[OK] TripoSG downloaded!')
else:
    print('[OK] TripoSG already downloaded.')

if not os.path.exists(RMBG_DIR):
    print('Downloading background removal model...')
    snapshot_download('briaai/RMBG-1.4', local_dir=RMBG_DIR)
    print('[OK] RMBG downloaded!')
else:
    print('[OK] RMBG already downloaded.')

print('\n All models ready!')


In [ ]:
#@title  Step 3 — Load models into memory { display-mode: "form" }
import sys, os, torch
sys.path.insert(0, '/content/TripoSG')
sys.path.insert(0, '/content/TripoSG/scripts')

from PIL import Image
import numpy as np
from briarmbg import BriaRMBG
from image_process import prepare_image
from triposg.pipelines.pipeline_triposg import TripoSGPipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16
print(f'Using device: {DEVICE}')

print('Loading background removal model...')
rmbg_net = BriaRMBG.from_pretrained('/content/checkpoints/RMBG-1.4').to(DEVICE)
rmbg_net.eval()

print('Loading TripoSG pipeline...')
pipe = TripoSGPipeline.from_pretrained('/content/checkpoints/TripoSG').to(DEVICE, DTYPE)

print('[OK] Models loaded and ready!')


In [ ]:
#@title  Step 4 — Launch LaunchMesh App { display-mode: "form" }
#@markdown Get a free ngrok token at https://ngrok.com then paste it below:
NGROK_TOKEN = ""  #@param {type:"string"}

import os, uuid, threading
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
import trimesh, torch
from PIL import Image

# Write HTML to a file to avoid string escaping issues
html_content = open("/content/TripoSG/../launchmesh_ui.html", "w") if False else None

HTML_FILE = "/content/launchmesh_ui.html"
with open(HTML_FILE, "w") as _f:
    _f.write("""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>LaunchMesh - Image to 3D</title>
<link href="https://fonts.googleapis.com/css2?family=Rajdhani:wght@500;600;700&family=Exo+2:wght@400;600;800&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{--bg:#040d1a;--surface:#071628;--surface2:#0a1e35;--border:#0e2d50;--orange:#f7931a;--orange2:#ffb347;--blue:#1e90ff;--blue2:#4db8ff;--text:#e8f4ff;--muted:#4a7a9b}
body{background:var(--bg);color:var(--text);font-family:"Exo 2",sans-serif;min-height:100vh;display:flex;flex-direction:column;align-items:center;justify-content:center;padding:40px 20px;overflow-x:hidden}
body::before{content:"";position:fixed;inset:0;background:radial-gradient(ellipse at 20% 10%,rgba(30,144,255,.12) 0%,transparent 40%),radial-gradient(ellipse at 80% 90%,rgba(247,147,26,.08) 0%,transparent 40%);pointer-events:none;z-index:0}
body::after{content:"";position:fixed;inset:0;background-image:radial-gradient(1px 1px at 10% 15%,rgba(255,255,255,.6) 0%,transparent 100%),radial-gradient(1px 1px at 25% 40%,rgba(255,255,255,.4) 0%,transparent 100%),radial-gradient(1px 1px at 60% 25%,rgba(255,255,255,.3) 0%,transparent 100%),radial-gradient(1px 1px at 75% 60%,rgba(255,255,255,.5) 0%,transparent 100%),radial-gradient(1px 1px at 85% 15%,rgba(255,255,255,.4) 0%,transparent 100%),radial-gradient(1px 1px at 15% 70%,rgba(255,255,255,.4) 0%,transparent 100%),radial-gradient(1px 1px at 50% 90%,rgba(255,255,255,.3) 0%,transparent 100%);pointer-events:none;z-index:0}
.container{position:relative;z-index:1;width:100%;max-width:660px}
header{margin-bottom:36px;text-align:center}
h1{font-family:"Rajdhani",sans-serif;font-size:clamp(2rem,7vw,3.2rem);font-weight:700;letter-spacing:.08em;text-transform:uppercase;line-height:1.1;margin-bottom:10px}
h1 .blue{color:var(--blue2)} h1 .orange{color:var(--orange)}
.subtitle{color:var(--muted);font-size:.82rem;letter-spacing:.18em;text-transform:uppercase;font-weight:600}
.badge{display:inline-block;background:rgba(247,147,26,.15);border:1px solid var(--orange);color:var(--orange);font-size:.65rem;padding:3px 10px;border-radius:20px;letter-spacing:.2em;font-weight:700;text-transform:uppercase;margin-top:10px}
.divider{height:1px;background:linear-gradient(90deg,transparent,var(--border),var(--blue),var(--border),transparent);margin:28px 0}
.drop-zone{border:1.5px dashed var(--border);border-radius:12px;padding:52px 40px;text-align:center;cursor:pointer;transition:all .25s ease;background:var(--surface);position:relative;overflow:hidden}
.drop-zone::before{content:"";position:absolute;inset:0;background:radial-gradient(ellipse at center,rgba(30,144,255,.06) 0%,transparent 70%);opacity:0;transition:opacity .25s ease}
.drop-zone:hover,.drop-zone.drag-over{border-color:var(--orange);box-shadow:0 0 30px rgba(247,147,26,.1);transform:translateY(-2px)}
.drop-zone:hover::before,.drop-zone.drag-over::before{opacity:1}
.drop-icon{font-size:2.8rem;margin-bottom:14px;display:block}
.drop-title{font-size:1.05rem;font-weight:700;margin-bottom:6px;font-family:"Rajdhani",sans-serif;letter-spacing:.05em;text-transform:uppercase}
.drop-hint{font-size:.78rem;color:var(--muted);letter-spacing:.1em}
input[type="file"]{display:none}
.preview-wrap{display:none;margin-top:20px;border-radius:10px;overflow:hidden;border:1px solid var(--border);position:relative}
.preview-wrap img{width:100%;max-height:280px;object-fit:contain;background:var(--surface);display:block}
.preview-label{position:absolute;top:10px;left:10px;background:rgba(247,147,26,.9);color:#040d1a;font-size:.65rem;padding:3px 8px;border-radius:4px;letter-spacing:.15em;font-weight:700;text-transform:uppercase}
.settings{margin-top:18px;background:var(--surface);border:1px solid var(--border);border-radius:10px;padding:18px 20px;display:none}
.settings-title{font-family:"Rajdhani",sans-serif;font-size:.75rem;letter-spacing:.2em;text-transform:uppercase;color:var(--muted);margin-bottom:14px}
.settings-grid{display:grid;grid-template-columns:1fr 1fr;gap:14px}
.setting-group{display:flex;flex-direction:column;gap:6px}
.setting-label{font-size:.68rem;color:var(--muted);letter-spacing:.15em;text-transform:uppercase;display:flex;justify-content:space-between}
.setting-label span{color:var(--orange2)}
input[type=range]{-webkit-appearance:none;width:100%;height:3px;background:var(--border);border-radius:2px;outline:none;cursor:pointer}
input[type=range]::-webkit-slider-thumb{-webkit-appearance:none;width:14px;height:14px;border-radius:50%;background:var(--orange);box-shadow:0 0 8px rgba(247,147,26,.5);cursor:pointer}
select{width:100%;padding:7px 10px;background:var(--surface2);border:1px solid var(--border);border-radius:6px;color:var(--text);font-family:"Exo 2",sans-serif;font-size:.8rem;outline:none;cursor:pointer}
.btn{display:block;width:100%;margin-top:18px;padding:15px;background:linear-gradient(135deg,var(--orange) 0%,var(--orange2) 100%);color:#040d1a;border:none;border-radius:8px;font-family:"Rajdhani",sans-serif;font-size:1.1rem;font-weight:700;letter-spacing:.12em;text-transform:uppercase;cursor:pointer;transition:all .2s ease;box-shadow:0 4px 20px rgba(247,147,26,.3)}
.btn:hover{transform:translateY(-2px);box-shadow:0 6px 28px rgba(247,147,26,.45)}
.btn:disabled{opacity:.35;cursor:not-allowed;transform:none;box-shadow:none}
.status-box{display:none;margin-top:20px;padding:18px 22px;background:var(--surface2);border:1px solid var(--border);border-radius:10px;font-size:.85rem}
.status-label{color:var(--muted);font-size:.65rem;letter-spacing:.25em;text-transform:uppercase;margin-bottom:6px;font-weight:600}
.status-text{color:var(--blue2);font-weight:600}
.status-note{color:var(--muted);font-size:.72rem;margin-top:6px;letter-spacing:.05em}
.progress-bar{height:3px;background:var(--border);border-radius:3px;margin-top:14px;overflow:hidden}
.progress-fill{height:100%;background:linear-gradient(90deg,var(--blue),var(--orange));width:0%;transition:width .5s ease;border-radius:3px;box-shadow:0 0 8px rgba(30,144,255,.6)}
.download-btn{display:none;margin-top:18px;padding:15px;background:transparent;color:var(--orange);border:1.5px solid var(--orange);border-radius:8px;font-family:"Rajdhani",sans-serif;font-size:1.1rem;font-weight:700;letter-spacing:.12em;text-transform:uppercase;cursor:pointer;text-align:center;text-decoration:none;transition:all .2s ease;width:100%;box-shadow:0 0 20px rgba(247,147,26,.1)}
.download-btn:hover{background:linear-gradient(135deg,var(--orange),var(--orange2));color:#040d1a;box-shadow:0 4px 24px rgba(247,147,26,.4);transform:translateY(-1px)}
footer{margin-top:36px;text-align:center;color:var(--muted);font-size:.72rem;letter-spacing:.15em;text-transform:uppercase}
footer a{color:var(--blue2);text-decoration:none}
footer a:hover{color:var(--orange)}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.4}}
.pulsing{animation:pulse 1.5s ease infinite}
@keyframes spin{to{transform:rotate(360deg)}}
.spinner{display:inline-block;width:14px;height:14px;border:2px solid rgba(4,13,26,.3);border-top-color:#040d1a;border-radius:50%;animation:spin .7s linear infinite;vertical-align:middle;margin-right:8px}
</style>
</head>
<body>
<div class="container">
  <header>
    <h1><span class="blue">Launch</span><span class="orange">Mesh</span></h1>
    <p class="subtitle">Image to 3D &middot; Powered by TripoSG</p>
    <span class="badge">&#9733; Free &middot; Community Edition</span>
  </header>
  <div class="divider"></div>
  <div class="drop-zone" id="dropZone">
    <span class="drop-icon">&#128640;</span>
    <div class="drop-title">Drop your image here</div>
    <div class="drop-hint">or click to browse &middot; PNG &middot; JPG &middot; WEBP</div>
    <input type="file" id="fileInput" accept="image/*">
  </div>
  <div class="preview-wrap" id="previewWrap">
    <span class="preview-label">Input</span>
    <img id="preview" src="" alt="Preview">
  </div>
  <div class="settings" id="settings">
    <div class="settings-title">&#9881; Generation Settings</div>
    <div class="settings-grid">
      <div class="setting-group">
        <div class="setting-label">Steps <span id="stepsVal">50</span></div>
        <input type="range" id="steps" min="10" max="100" value="50" oninput="document.getElementById('stepsVal').textContent=this.value">
      </div>
      <div class="setting-group">
        <div class="setting-label">Guidance <span id="guidanceVal">7.5</span></div>
        <input type="range" id="guidance" min="1" max="15" step="0.5" value="7.5" oninput="document.getElementById('guidanceVal').textContent=this.value">
      </div>
      <div class="setting-group">
        <div class="setting-label">Simplify</div>
        <select id="simplify">
          <option value="0.95">0.95 - High detail</option>
          <option value="0.90" selected>0.90 - Balanced</option>
          <option value="0.80">0.80 - Lighter mesh</option>
        </select>
      </div>
      <div class="setting-group">
        <div class="setting-label">Seed</div>
        <select id="seedMode">
          <option value="random" selected>Random</option>
          <option value="fixed">Fixed (42)</option>
        </select>
      </div>
    </div>
  </div>
  <button class="btn" id="convertBtn" disabled>&#9889; Generate 3D Model</button>
  <div class="status-box" id="statusBox">
    <div class="status-label">Status</div>
    <div class="status-text" id="statusText">Initializing...</div>
    <div class="status-note">TripoSG runs on Colab GPU - processing takes 1-3 minutes.</div>
    <div class="progress-bar"><div class="progress-fill" id="progressFill"></div></div>
  </div>
  <a class="download-btn" id="downloadBtn" href="#" download="launchmesh_model.glb">&#8681; Download .glb Model</a>
  <div class="divider" style="margin-top:32px"></div>
  <footer>
    <a href="https://www.skool.com/miless-group-4588" target="_blank">3D Launchpad Community</a>
    &middot; TripoSG by VAST-AI
  </footer>
</div>
<script>
const dropZone=document.getElementById("dropZone"),fileInput=document.getElementById("fileInput"),preview=document.getElementById("preview"),previewWrap=document.getElementById("previewWrap"),convertBtn=document.getElementById("convertBtn"),statusBox=document.getElementById("statusBox"),statusText=document.getElementById("statusText"),progressFill=document.getElementById("progressFill"),downloadBtn=document.getElementById("downloadBtn"),settings=document.getElementById("settings");
let selectedFile=null;
dropZone.addEventListener("click",()=>fileInput.click());
dropZone.addEventListener("dragover",e=>{e.preventDefault();dropZone.classList.add("drag-over");});
dropZone.addEventListener("dragleave",()=>dropZone.classList.remove("drag-over"));
dropZone.addEventListener("drop",e=>{e.preventDefault();dropZone.classList.remove("drag-over");const f=e.dataTransfer.files[0];if(f&&f.type.startsWith("image/"))loadFile(f);});
fileInput.addEventListener("change",()=>{if(fileInput.files[0])loadFile(fileInput.files[0]);});
function loadFile(f){selectedFile=f;preview.src=URL.createObjectURL(f);previewWrap.style.display="block";settings.style.display="block";convertBtn.disabled=false;downloadBtn.style.display="none";statusBox.style.display="none";}
convertBtn.addEventListener("click",async()=>{
  if(!selectedFile)return;
  convertBtn.disabled=true;convertBtn.innerHTML='<span class="spinner"></span>Generating...';
  downloadBtn.style.display="none";statusBox.style.display="block";progressFill.style.width="5%";
  statusText.textContent="Uploading image...";statusText.classList.add("pulsing");
  const seed=document.getElementById("seedMode").value==="random"?Math.floor(Math.random()*2147483647):42;
  const fd=new FormData();
  fd.append("image",selectedFile);
  fd.append("steps",document.getElementById("steps").value);
  fd.append("guidance",document.getElementById("guidance").value);
  fd.append("simplify",document.getElementById("simplify").value);
  fd.append("seed",seed);
  try{
    progressFill.style.width="15%";statusText.textContent="Processing image...";
    const resp=await fetch("/generate",{method:"POST",body:fd});
    if(!resp.ok)throw new Error(await resp.text());
    progressFill.style.width="40%";statusText.textContent="Running TripoSG... (1-3 min)";
    const data=await resp.json();const jobId=data.job_id;
    let done=false,attempts=0;
    const labels=["Removing background...","Encoding features...","Running diffusion...","Extracting mesh...","Simplifying mesh...","Finalising..."];
    while(!done){
      await new Promise(r=>setTimeout(r,3000));
      const poll=await fetch("/status/"+jobId);const s=await poll.json();
      if(s.status==="done"){done=true;progressFill.style.width="100%";statusText.textContent="Complete!";statusText.classList.remove("pulsing");downloadBtn.href="/download/"+jobId;downloadBtn.style.display="block";convertBtn.innerHTML="&#9889; Generate 3D Model";convertBtn.disabled=false;}
      else if(s.status==="error"){throw new Error(s.message||"Generation failed");}
      else{attempts++;progressFill.style.width=Math.min(40+attempts*8,90)+"%";statusText.textContent=labels[Math.min(attempts-1,labels.length-1)];}
    }
  }catch(err){statusText.textContent="Error: "+err.message;statusText.classList.remove("pulsing");statusText.style.color="#f87171";convertBtn.innerHTML="&#9889; Generate 3D Model";convertBtn.disabled=false;}
});
</script>
</body>
</html>""")

# Flask app
flask_app = Flask(__name__)
CORS(flask_app)
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
jobs = {}

@flask_app.route("/")
def index():
    return open(HTML_FILE).read()

@flask_app.route("/generate", methods=["POST"])
def generate():
    import random
    if "image" not in request.files:
        return jsonify({"error": "No image"}), 400
    f        = request.files["image"]
    steps    = int(request.form.get("steps", 50))
    guidance = float(request.form.get("guidance", 7.5))
    simplify = float(request.form.get("simplify", 0.90))
    seed     = int(request.form.get("seed", random.randint(0, 2147483647)))
    job_id   = str(uuid.uuid4())[:8]
    img_path = f"/content/outputs/{job_id}_input.png"
    out_path = f"/content/outputs/{job_id}.glb"
    f.save(img_path)
    jobs[job_id] = {"status": "running", "out_path": out_path}

    def run_triposg():
        try:
            img = Image.open(img_path).convert("RGBA")
            processed = prepare_image(img, rmbg_net, DEVICE)
            torch.manual_seed(seed)
            outputs = pipe(
                image=processed,
                num_inference_steps=steps,
                guidance_scale=guidance,
            ).samples[0]
            vertices, faces = outputs
            mesh = trimesh.Trimesh(vertices.cpu().numpy(), faces.cpu().numpy())
            mesh = mesh.simplify_quadric_decimation(int(len(mesh.faces) * simplify))
            mesh.export(out_path)
            jobs[job_id]["status"] = "done"
        except Exception as e:
            jobs[job_id] = {"status": "error", "message": str(e)}
            print(f"[ERROR] {e}")

    threading.Thread(target=run_triposg, daemon=True).start()
    return jsonify({"job_id": job_id})

@flask_app.route("/status/<job_id>")
def status(job_id):
    return jsonify(jobs.get(job_id, {"status": "unknown"}))

@flask_app.route("/download/<job_id>")
def download(job_id):
    job = jobs.get(job_id)
    if not job or job["status"] != "done":
        return "Not ready", 404
    return send_file(job["out_path"], as_attachment=True, download_name="launchmesh_model.glb")

# Start ngrok and server
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(5000)
print(f"\n[OK] LaunchMesh is LIVE!")
print(f"\n>> Open this link: {public_url}")
print(f"\n   Share with your community!")

flask_app.run(port=5000, use_reloader=False)


---
## 💡 Tips for best results
- **Clean background** — white or solid colour behind the object
- **Single object** — one item centred in frame
- **Good lighting** — evenly lit, minimal shadows
- **Square crop** — crop tight around the object

## 📁 Output
Downloads as a `.glb` file — open in **Blender** (File → Import → glTF 2.0), Windows 3D Viewer, Unity, or Unreal.

## ⚡ Free GPU limits
Each person who opens their own copy of this notebook gets their own independent Colab GPU session — no shared quota!

## 🔗 Your shareable link
```
https://colab.research.google.com/github/chopsitv/launchmesh-triposg/blob/main/LaunchMesh_TripoSG.ipynb
```